In [1]:
import pandas as pd

# 1. Load the real Caltech ACN sessions dataset
df = pd.read_csv("ACN_sessions.csv")

# 2. Check the exact number of rows and columns
print("Rows and Columns:", df.shape)

# 3. Check all exact column names
print("\nColumn Names:", list(df.columns))

# 4. Check for missing values in each column
print("\nMissing Values:\n", df.isnull().sum())

# 5. Preview the first 3 rows
print("\nFirst 3 Rows:\n", df.head(3))


Rows and Columns: (2026, 13)

Column Names: ['_id', 'userInputs', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'connectionTime', 'disconnectTime', 'kWhDelivered', 'doneChargingTime', 'timezone', 'userID']

Missing Values:
 _id                    0
userInputs          2013
sessionID              0
stationID              0
spaceID                0
siteID                 0
clusterID              0
connectionTime         0
disconnectTime         0
kWhDelivered           0
doneChargingTime       1
timezone               0
userID              2013
dtype: int64

First 3 Rows:
                         _id userInputs  \
0  5bc90cb9f9af8b0d7fe77cd2        NaN   
1  5bc90cb9f9af8b0d7fe77cd3        NaN   
2  5bc90cb9f9af8b0d7fe77cd4        NaN   

                                sessionID    stationID spaceID  siteID  \
0  2_39_78_362_2018-04-25 11:08:04.400812  2-39-78-362  CA-496       2   
1   2_39_95_27_2018-04-25 13:45:09.617470   2-39-95-27  CA-319       2   
2  2_39_79_380_20

In [2]:
import pandas as pd
import numpy as np

# 1. Load the real dataset
df = pd.read_csv("ACN_sessions.csv")

# 2. Drop columns that are >99% empty
df_clean = df.drop(columns=["userInputs", "userID"]).copy()

# 3. Fill the 1 missing doneChargingTime with disconnectTime
df_clean["doneChargingTime"] = df_clean["doneChargingTime"].fillna(df_clean["disconnectTime"])

# 4. Parse GMT strings and convert to local Pasadena time (America/Los_Angeles)
time_cols = ["connectionTime", "disconnectTime", "doneChargingTime"]
for col in time_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], utc=True).dt.tz_convert("America/Los_Angeles")

# 5. Calculate exact durations in minutes (with logical safeguards against sensor clock drift)
df_clean["Total_Connected_Mins"] = (df_clean["disconnectTime"] - df_clean["connectionTime"]).dt.total_seconds() / 60.0
raw_idle_mins = (df_clean["disconnectTime"] - df_clean["doneChargingTime"]).dt.total_seconds() / 60.0

# Clamp idle minutes between 0 and Total_Connected_Mins so negative numbers are impossible
df_clean["Idle_Mins"] = np.clip(raw_idle_mins, 0, df_clean["Total_Connected_Mins"])
df_clean["Active_Charging_Mins"] = df_clean["Total_Connected_Mins"] - df_clean["Idle_Mins"]

# 6. Extract clean local time features
df_clean["Arrival_Hour"] = df_clean["connectionTime"].dt.hour
df_clean["Day_Number"] = df_clean["connectionTime"].dt.dayofweek  # 0=Monday, 6=Sunday
df_clean["Day_of_Week"] = df_clean["connectionTime"].dt.day_name()

# 7. Logical Integrity Verification
print("Missing values remaining:", df_clean.isnull().sum().sum())
print("Any negative connected times?", (df_clean["Total_Connected_Mins"] <= 0).any())
print("Any negative idle times?", (df_clean["Idle_Mins"] < 0).any())
print("\n--- Clean Metrics Summary (Minutes & kWh) ---")
print(df_clean[["Total_Connected_Mins", "Active_Charging_Mins", "Idle_Mins", "kWhDelivered"]].describe().round(2))


Missing values remaining: 0
Any negative connected times? False
Any negative idle times? False

--- Clean Metrics Summary (Minutes & kWh) ---
       Total_Connected_Mins  Active_Charging_Mins  Idle_Mins  kWhDelivered
count               2026.00               2026.00    2026.00       2026.00
mean                 378.81                210.38     168.43          8.85
std                  369.51                191.86     318.77          6.72
min                    5.73                  0.00       0.00          0.52
25%                  143.32                 81.51       0.08          4.00
50%                  326.01                150.89      42.58          7.25
75%                  536.24                274.10     274.42         12.65
max                 5075.67               1824.27    4931.58         47.81


In [3]:
# 1. Top 5 Busiest Local Arrival Hours (Pasadena Time)
hourly_stats = df_clean.groupby("Arrival_Hour").agg(
    Total_Sessions=("sessionID", "count"),
    Avg_Connected_Mins=("Total_Connected_Mins", "mean"),
    Avg_Idle_Mins=("Idle_Mins", "mean"),
    Total_kWh=("kWhDelivered", "sum")
).round(1)

print("--- Top 5 Busiest Arrival Hours (Local Pasadena Time) ---")
print(hourly_stats.sort_values(by="Total_Sessions", ascending=False).head(5))

# 2. Top 5 Most Congested Physical Chargers (stationID)
station_stats = df_clean.groupby(["stationID", "spaceID"]).agg(
    Total_Sessions=("sessionID", "count"),
    Total_Connected_Hrs=("Total_Connected_Mins", lambda x: round(x.sum() / 60, 1)),
    Total_Idle_Hrs=("Idle_Mins", lambda x: round(x.sum() / 60, 1)),
    Total_kWh=("kWhDelivered", "sum")
).round(1)

print("\n--- Top 5 Most Congested Physical Charging Bays ---")
print(station_stats.sort_values(by="Total_Sessions", ascending=False).head(5))


--- Top 5 Busiest Arrival Hours (Local Pasadena Time) ---
              Total_Sessions  Avg_Connected_Mins  Avg_Idle_Mins  Total_kWh
Arrival_Hour                                                              
8                        416               455.3          225.3     3346.3
9                        283               411.5          151.5     2576.3
7                        139               480.3          225.1     1373.1
10                       134               386.7          133.5     1278.1
18                       127               277.6          101.3     1087.0

--- Top 5 Most Congested Physical Charging Bays ---
                     Total_Sessions  Total_Connected_Hrs  Total_Idle_Hrs  \
stationID   spaceID                                                        
2-39-139-28 CA-303               85                490.5           134.4   
2-39-138-29 CA-304               74                259.1            79.3   
2-39-127-19 CA-309               66                282.0    

In [4]:
# 1. Final Edge-Case Verification (1-to-1 Station Mapping & Weekly Pattern)
print("Unique stationID count:", df_clean["stationID"].nunique())
print("Unique spaceID count:", df_clean["spaceID"].nunique())

print("\n--- Weekly Congestion Pattern (Calendar Sorted) ---")
weekly_stats = df_clean.groupby(["Day_Number", "Day_of_Week"]).agg(
    Total_Sessions=("sessionID", "count"),
    Avg_Connected_Mins=("Total_Connected_Mins", "mean"),
    Avg_Idle_Mins=("Idle_Mins", "mean"),
    Total_kWh=("kWhDelivered", "sum")
).round(1)
print(weekly_stats)

# 2. Prepare clean export copy & strip timezone offset so Power BI keeps local Pasadena time
export_df = df_clean.copy()
for col in ["connectionTime", "disconnectTime", "doneChargingTime"]:
    export_df[col] = export_df[col].dt.tz_localize(None)

# 3. Add session-level Idle Ratio (%) and Hogging Category for Power BI slicers
export_df["Idle_Ratio_Pct"] = ((export_df["Idle_Mins"] / export_df["Total_Connected_Mins"]) * 100).round(1)
export_df["Hogging_Category"] = np.where(
    export_df["Idle_Ratio_Pct"] == 0, "Zero Idle",
    np.where(export_df["Idle_Ratio_Pct"] <= 50, "Moderate Idle (1-50%)", "Severe Hogging (>50%)")
)

# 4. Final Assertion Check before saving
assert export_df.isnull().sum().sum() == 0, "Null values detected before export!"
assert len(export_df) == 2026, "Row count changed!"

# 5. Export to CSV for Power BI Desktop
export_df.to_csv("ACN_Cleaned_Real.csv", index=False)
print("\nExported successfully! Rows and Columns:", export_df.shape)
print("Columns ready for Power BI:", list(export_df.columns))


Unique stationID count: 51
Unique spaceID count: 51

--- Weekly Congestion Pattern (Calendar Sorted) ---
                        Total_Sessions  Avg_Connected_Mins  Avg_Idle_Mins  \
Day_Number Day_of_Week                                                      
0          Monday                  268               381.1          156.7   
1          Tuesday                 325               394.2          173.8   
2          Wednesday               405               388.5          188.4   
3          Thursday                389               368.5          168.0   
4          Friday                  327               436.0          198.8   
5          Saturday                177               328.3          137.0   
6          Sunday                  135               265.8           88.1   

                        Total_kWh  
Day_Number Day_of_Week             
0          Monday          2377.4  
1          Tuesday         2818.9  
2          Wednesday       3430.8  
3          Thursday  